# Notebook 02: Feature Engineering, VIF Analysis & PCA
**Author:** Mark Eliezer M. Villola | ProTech Devices Philippines | AIM Capstone 2025

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')
from fraud_detection_pipeline import engineer_features, clean_data, drop_high_missing_features
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
print('Environment ready.')

## 1. Load EDA Output

In [ ]:
try:
    df = pd.read_parquet('../data/processed/01_eda_output.parquet')
    print(f'Loaded: {df.shape}')
except FileNotFoundError:
    from fraud_detection_pipeline import generate_demo_data
    df = generate_demo_data(n_samples=40000)
    df = clean_data(df)

## 2. Feature Engineering — 10 InsurTech Signals

In [ ]:
df = engineer_features(df)
eng_feats = ['log_claim_amt','days_since_policy','is_early_claim','claim_velocity',
             'email_risk_score','amt_to_device_ratio','mismatch_score',
             'high_risk_x_early','log_dist1','amt_zscore']
print('Engineered features:')
for f in eng_feats:
    if f in df.columns:
        print(f'  {f:<30} | null: {df[f].isnull().sum()} | mean: {df[f].mean():.4f}')

## 3. Feature Selection via XGBoost Importance

In [ ]:
from fraud_detection_pipeline import select_features
selected = select_features(df, target='isFraud', max_features=55)
print(f'Selected {len(selected)} features')
os.makedirs('../data/processed', exist_ok=True)
import json; open('../data/processed/selected_features.json','w').write(json.dumps(selected))
print('Feature list saved.')

## 4. PCA on V-Features

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
v_cols = [c for c in df.columns if c.startswith('V') and c[1:].isdigit() and c in selected]
if len(v_cols) >= 5:
    X_v = df[v_cols].fillna(0)
    X_vs = StandardScaler().fit_transform(X_v)
    pca = PCA(n_components=min(30, len(v_cols)))
    pca.fit(X_vs)
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    n95 = np.argmax(cum_var >= 0.95) + 1
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(range(1,len(cum_var)+1), cum_var*100, 'o-', color='#2563EB', linewidth=2)
    ax.axhline(95, color='red', linestyle='--', label='95% threshold')
    ax.axvline(n95, color='orange', linestyle='--', label=f'{n95} components')
    ax.set_title('PCA — Cumulative Explained Variance (V-Features)', fontweight='bold')
    ax.set_xlabel('Principal Components'); ax.set_ylabel('Cumulative Variance (%)')
    ax.legend(); plt.tight_layout()
    os.makedirs('../reports/figures',exist_ok=True)
    plt.savefig('../reports/figures/02_pca_variance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'{len(v_cols)} V-features -> {n95} PCA components explain 95% of variance')
else:
    print('Insufficient V-features for PCA in demo mode.')

## 5. Save Processed Feature Set

In [ ]:
TARGET = 'isFraud'
keep = [c for c in selected if c in df.columns] + [TARGET]
df[keep].to_parquet('../data/processed/02_features_output.parquet', index=False)
print(f'Saved: {len(keep)-1} features + target')